# comms.io daily run

This is the Colab equivalent of the old `skeb-archiver` notebook, adapted to
commits.io:

- clones the **master** (archiver) branch and a sparse copy of the **data** branch
- restores the SQLite working database from `data/state/`
- runs the archiver and publishes the updated data branch
- uses **no Google Drive**

Add `GITHUB_TOKEN` in the Colab secrets panel with `contents: write`.
The website reads the data branch through raw.githubusercontent.com, so a data
publish does not need a Pages rebuild.

In [ ]:
# Setup
from pathlib import Path
import os
import subprocess
import sys

MAIN_DIR = "comms.io"
DATA_DIR = "data"
REPO = "Elfiawesome/comms.io"
REPO_URL = f"https://github.com/{REPO}.git"
DATA_BRANCH = "data"

# Run mode: daily | refresh | seeds | gallery-dl | status
RUN_MODE = "daily"
LIMIT = 0                 # 0 = let the config request/runtime budgets stop the run
MAX_RUNTIME = 180         # minutes
SEED_FILE = ""            # local path or raw URL, e.g. "seeds/skeb_album.md"

%cd /content
!rm -rf {MAIN_DIR} {DATA_DIR}

os.environ["COMMS_TRIGGER_DEPLOY"] = "false"
os.environ["COMMS_DATABASE"] = f"/content/{MAIN_DIR}/work/comms.sqlite3"
os.environ["COMMS_EXPORT_DIR"] = f"/content/{MAIN_DIR}/dist/data/v1"
os.environ["COMMS_GALLERY_DL_DIR"] = f"/content/{MAIN_DIR}/work/gallery-dl"

print("main:", MAIN_DIR)
print("data:", DATA_DIR)
print("mode:", RUN_MODE)

In [ ]:
# Token and branch setup
from google.colab import userdata

GHPAT = userdata.get("GITHUB_TOKEN")
os.environ["GITHUB_TOKEN"] = GHPAT

# Avoid putting the token in clone URLs or notebook output.
askpass = Path("/content/askpass.sh")
askpass.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    '  *Username*) echo x-access-token ;;\n'
    '  *) echo "$GITHUB_TOKEN" ;;\n'
    'esac\n',
    encoding="utf-8",
)
askpass.chmod(0o700)
os.environ["GIT_ASKPASS"] = str(askpass)
os.environ["GIT_TERMINAL_PROMPT"] = "0"

# Clone the archiver branch.
!git clone -b master --depth=1 {REPO_URL} {MAIN_DIR}

# Clone only state/ from the data branch. The website Parquet data is fetched
# by the website directly from raw.githubusercontent.com.
!git clone -b {DATA_BRANCH} --depth=1 --filter=blob:none --sparse {REPO_URL} {DATA_DIR}
!git -C {DATA_DIR} sparse-checkout set state

print("token loaded:", bool(GHPAT))

In [ ]:
# Local venv and package install
%cd /content/{MAIN_DIR}

venv = Path(".venv")
if not (venv / "bin" / "python").exists():
    !{sys.executable} -m venv .venv

PY = str((venv / "bin" / "python").resolve())
os.environ["PATH"] = str((venv / "bin").resolve()) + os.pathsep + os.environ.get("PATH", "")
!{PY} -m pip install --upgrade pip
!{PY} -m pip install -e .

if RUN_MODE == "gallery-dl":
    !{PY} -m pip install gallery-dl

In [ ]:
# Restore the full SQLite state from the data branch. If there is no state
# snapshot yet, the archiver starts with an empty database.
DATA_BASE = Path(f"/content/{DATA_DIR}").as_uri() + "/"
!{PY} -m comms state unpack --repo {REPO} --branch {DATA_BRANCH} --data-base {DATA_BASE} --database /content/{MAIN_DIR}/work/comms.sqlite3

In [ ]:
# Run the archiver. --publish exports Parquet + repacks state + pushes data.
run_args = {
    "daily": f"--limit {LIMIT} --max-runtime {MAX_RUNTIME} --publish",
    "refresh": f"--no-discover --refresh --limit {LIMIT} --max-runtime {MAX_RUNTIME} --publish",
    "seeds": f'--no-discover --no-refresh --file "{SEED_FILE}" --max-runtime {MAX_RUNTIME} --publish',
    "gallery-dl": f"--run-gallery-dl --limit {LIMIT} --max-runtime {MAX_RUNTIME} --publish",
    "status": "",
}

if RUN_MODE not in run_args:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")
if RUN_MODE == "seeds" and not SEED_FILE:
    raise ValueError("Set SEED_FILE before using seeds mode.")

if RUN_MODE == "status":
    !{PY} -m comms status
else:
    print(f"running: comms crawl {run_args[RUN_MODE]}")
    !{PY} -m comms crawl {run_args[RUN_MODE]}

In [ ]:
# Inspect the result.
!{PY} -m comms status
!{PY} -m comms validate
!{PY} -m comms state status

## Modes

- `daily` — discovery + refresh. The normal morning run.
- `refresh` — refresh stale artists without listing discovery.
- `seeds` — import a markdown/CSV/JSON seed file; set `SEED_FILE`.
- `gallery-dl` — run gallery-dl first, then import its X metadata.
- `status` — print database and remote state counts without crawling.

## Notes

- The data branch is force-pushed as a squashed snapshot. Do not edit it by hand.
- The website reads the new Parquet data immediately through raw.githubusercontent.com.
- If repeated 429s occur, lower `network.requests_per_second` and concurrency in `config/comms.toml`.
- The old Discord webhook is intentionally not hard-coded. Use a Colab secret if you add notifications later.